In [2]:
import numpy as np
import pandas as pd
import gc
from tqdm import tqdm  # Ensure tqdm is imported

In [3]:
def clean_dataframe(df):
    print("Initial shape:", df.shape)
    df_cleaned = df.dropna(how='all', axis=1)
    print("Shape after dropping columns with all NaN values:", df_cleaned.shape)
    df_cleaned = df_cleaned.dropna(how='all', axis=0)
    print("Shape after dropping rows with all NaN values:", df_cleaned.shape)

    return df_cleaned


def reduce_memory_usage(df):
    start_mem = df.memory_usage().sum() / 1024 ** 2
    shape1 = df.shape
    print(f'Initial memory usage: {start_mem:.2f} MB')

    for col in tqdm(df.columns, desc="Optimizing columns"):
        col_type = df[col].dtype

        if "int" in str(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        elif "float" in str(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)
        elif col_type == object:
            # Attempt to identify date-like strings
            if df[col].str.contains('-', na=False).any() or df[col].str.contains('/', na=False).any():
                df_temp = pd.to_datetime(df[col], errors='coerce')
                # Check if a significant portion of the column was successfully converted
                if not df_temp.isnull().all() and df_temp.notnull().sum() / len(df_temp) > 0.5:
                    df[col] = df_temp
                else:
                    # Convert to string if datetime conversion is largely unsuccessful
                    df[col] = df[col].astype(str)
            else:
                # Convert to string if no date-like patterns are detected
                df[col] = df[col].astype(str)
    print("...inferring objects...")
    df = df.infer_objects()

    end_mem = df.memory_usage().sum() / 1024 ** 2
    print(f'Memory usage after optimization: {end_mem:.2f} MB')
    print(f'Decreased by {(100 * (start_mem - end_mem) / start_mem):.2f}%')
    shape2 = df.shape
    print(f'...Shape changed from {shape1} to {shape2}')
    gc.collect()
    return df

In [4]:
df = pd.read_csv(
    r"E:\PHD One Drive\OneDrive - University of Edinburgh\PHD_OUTPUT\2024-10 RETfound features\output_imagenet\features.csv")
df.head(1)

,image_name,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_1014,feature_1015,feature_1016,feature_1017,feature_1018,feature_1019,feature_1020,feature_1021,feature_1022,feature_1023
0,1000040_21015_1_0.png,0.010418,0.078763,-0.206168,0.552051,0.945177,-0.52902,-0.292406,-0.037438,-0.082797,...,0.075221,0.151138,-0.575239,-0.408544,0.000541,0.674464,0.351752,-0.310307,0.807674,0.092432


In [ ]:
df1 = clean_dataframe(df)